In [ ]:
import pandas as pd

# Ganti 'nama_file_anda.csv' dengan nama file yang baru Anda upload
df = pd.read_csv('/content/ulasan_com.ss.android.ugc.trill.csv')

# Menampilkan informasi jumlah baris dan kolom
print(f"Jumlah data: {df.shape[0]} baris")

# Menampilkan 5 data teratas untuk melihat isi kolomnya
df.head()

Jumlah data: 100000 baris


,Nama User,Ulasan,Rating,Tanggal,Likes,Versi App
0,Pengguna Google,aplikasi nya seru banget minn aku suka sering ...,5,2025-12-31 17:31:32,0,43.1.4
1,Pengguna Google,susah mulung skrg,5,2025-12-31 17:28:37,0,43.1.4
2,Pengguna Google,tik tok rusak yg pada ngomong kasar di laporka...,1,2025-12-31 17:25:28,0,40.1.4
3,Pengguna Google,seru,4,2025-12-31 17:23:56,0,NaN
4,Pengguna Google,tiktok ny ga bisa di cptin suka nykrol sendiri...,1,2025-12-31 17:21:25,0,43.1.4


In [ ]:
import re

# 1. Membuang baris yang ulasannya kosong (NaN)
df = df.dropna(subset=['Ulasan'])

# 2. Membuat label Sentimen berdasarkan Rating
# Rating 1-2 = Sentimen Negatif (0)
# Rating 4-5 = Sentimen Positif (1)
# Rating 3 kita buang (dianggap netral) agar model lebih tajam perbedaannya
df_clean = df[df['Rating'] != 3].copy()
df_clean['Sentimen'] = df_clean['Rating'].apply(lambda x: 1 if x > 3 else 0)

# 3. Fungsi dasar untuk membersihkan teks (Text Cleaning)
def bersihkan_teks(teks):
    teks = str(teks).lower() # Ubah ke huruf kecil
    teks = re.sub(r'[^a-z\s]', '', teks) # Hapus angka, emoji, dan tanda baca
    teks = teks.strip() # Hapus spasi berlebih di awal/akhir
    return teks

# Terapkan fungsi pembersih ke kolom ulasan
df_clean['Ulasan_Bersih'] = df_clean['Ulasan'].apply(bersihkan_teks)

# Tampilkan hasilnya
print(f"Sisa data setelah dibersihkan: {df_clean.shape[0]} baris")
df_clean[['Ulasan', 'Ulasan_Bersih', 'Rating', 'Sentimen']].head()

Sisa data setelah dibersihkan: 92740 baris


,Ulasan,Ulasan_Bersih,Rating,Sentimen
0,aplikasi nya seru banget minn aku suka sering ...,aplikasi nya seru banget minn aku suka sering ...,5,1
1,susah mulung skrg,susah mulung skrg,5,1
2,tik tok rusak yg pada ngomong kasar di laporka...,tik tok rusak yg pada ngomong kasar di laporka...,1,0
3,seru,seru,4,1
4,tiktok ny ga bisa di cptin suka nykrol sendiri...,tiktok ny ga bisa di cptin suka nykrol sendiri...,1,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# 1. Membagi data: 80% untuk AI belajar (Training), 20% untuk ujian (Testing)
X = df_clean['Ulasan_Bersih']
y = df_clean['Sentimen']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Mengubah teks menjadi angka (TF-IDF)
# Kita batasi mengambil 5000 kata yang paling sering muncul agar prosesnya ringan
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 3. Melatih Model AI dengan algoritma SVM
print("AI sedang belajar dari puluhan ribu ulasan, mohon tunggu sebentar...")
model_svm = LinearSVC(random_state=42)
model_svm.fit(X_train_tfidf, y_train)
print("Proses belajar selesai! ✅\n")

# 4. Mengevaluasi Model (Ujian seberapa pintar AI-nya)
prediksi = model_svm.predict(X_test_tfidf)
akurasi = accuracy_score(y_test, prediksi)

print(f"🎯 Akurasi Model: {akurasi * 100:.2f}%\n")
print("📊 Laporan Detail Evaluasi (Modul 5):")
print(classification_report(y_test, prediksi, target_names=['Negatif', 'Positif']))

AI sedang belajar dari puluhan ribu ulasan, mohon tunggu sebentar...
Proses belajar selesai! ✅

🎯 Akurasi Model: 84.88%

📊 Laporan Detail Evaluasi (Modul 5):
              precision    recall  f1-score   support

     Negatif       0.79      0.82      0.81      7094
     Positif       0.89      0.87      0.88     11454

    accuracy                           0.85     18548
   macro avg       0.84      0.84      0.84     18548
weighted avg       0.85      0.85      0.85     18548



In [ ]:
import joblib

# Menyimpan model AI
joblib.dump(model_svm, 'model_sentimen_svm.pkl')

# Menyimpan alat penerjemah teks (TF-IDF)
joblib.dump(vectorizer, 'vectorizer_tfidf.pkl')

print("Model dan Vectorizer berhasil disimpan! 💾")

Model dan Vectorizer berhasil disimpan! 💾
